<!-- KI: Claude, Zweck: eine Analyseseite von den Parametern bis zur Empfehlung im Design des Notebook-Prototyps (E-053), geprüft von allen Ownern R1–R4 -->

# Reinigung in die günstigsten Nachtstunden legen

**Leitfrage:** In welchem Maße kann ein bayerisches Molkereiwerk (anonym) seine Stromkosten am deutschen Day-Ahead-Markt senken, wenn es die nächtliche Maschinenreinigung in die günstigsten Stunden legt, deren Zeitpunkt jedoch eine Woche im Voraus im Schichtplan feststehen muss (2022–2026)?

**Fragestellungen (Teamdokument):** FS1 feste neue Uhrzeit aus den historischen Daten · FS2 jede Woche am Freitag aus den letzten 16 Wochen neu planen · FS3 hätten wir vor einem Jahr so geplant: gespart?

Diese Seite rechnet nichts neu. Sie liest die geprüften Ergebnisse der Pipeline (`python src/run_all.py`, 80 Tests) und erklärt sie der Reihe nach. Jede Zahl im Text wird beim Ausführen aus den Ergebnisdateien eingesetzt. Owner je Abschnitt in Klammern.

In [ ]:
import sys, importlib.util
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, Image, display

sys.path.insert(0, "src")
from gemeinsam import ROOT, lade_params, pfad

def modul(datei):
    spec = importlib.util.spec_from_file_location(datei, ROOT / "src" / f"{datei}.py")
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m

br = modul("07_bericht")
de = br.de                                   # German number format
p = lade_params()
T, R2 = pfad(p, "tables"), ROOT / p["cleaning"]["reports"]
lies = lambda name, ordner=None: pd.read_csv((ordner or T) / name)
zeige = lambda name: display(Image(filename=str(T / "grafiken" / name)))
text = lambda t: display(Markdown(t))
pd.set_option("display.float_format", "{:,.2f}".format);   # no 2.9e+01 in tables
pd.set_option("display.max_columns", 20); pd.set_option("display.width", 200)

## 0 · Parameter

Alle Annahmen stehen in `params.yaml`, mit Quelle je Wert; die vollständige Liste mit Art (Partner, Schätzung, Team, Daten) steht in `docs/annahmen.md`.

In [ ]:
L, K, V = p["last"], p["kosten"], p["vertrag"]
pd.DataFrame([
    ("Produktion", f"{L['betrieb_start_h']:02d}–{L['betrieb_ende_h']:02d} Uhr, jeden Tag", "Partner"),
    ("Reinigungsfenster", f"{L['fenster_start_h']:02d}–{L['fenster_ende_h']:02d} Uhr", "Partner"),
    ("Status quo A", f"{L['status_quo_start_h']}–{L['status_quo_start_h'] + L['blocklaenge_h']} Uhr", "Partner (K-012)"),
    ("Reinigung", f"{L['mwh_pro_tag']:g} MWh in {L['blocklaenge_h']} h (4 MW)", "Partner (E-051)"),
    ("Verbrauch Werk", f"{L['gesamtverbrauch_mwh_pro_tag']:g} MWh/Tag, davon {V['spot_anteil']:.0%} Day-Ahead", "Partner"),
    ("Festpreis", f"{K['festpreis_eur_mwh']:g} €/MWh", "Partner"),
    ("Personal", f"{K['mitarbeiter']} Personen à {K['stundenlohn_eur']:g} €/h", "Teamannahme"),
    ("Zuschläge", f"Nacht {K['nachtzuschlag']:.0%} (22–06), Wochenende {K['wochenendzuschlag']:.0%}, Feiertag {K['feiertagszuschlag']:.0%}; {K['zuschlag_regel']}", "Partner geschätzt, Regel Team"),
    ("Planung", f"Freitag {p['planung']['planungsuhrzeit']} für Mo–So der Folgewoche", "Partner"),
    ("Zeiträume", "Training 2023–2024 · Test 1 01–09/2025 · Test 2 10/2025–09/2026 · Stresstest 2022", "Team"),
    ("Statistik", f"α = {p['statistik']['alpha_gesamt']} / {p['statistik']['anzahl_hypothesen']} (Bonferroni), Seed {p['seed']}", "Team"),
], columns=["Annahme", "Wert", "Quelle"])

## 1 · Daten laden und aufbereiten (R2 Max)

### 1a · Quelle
Day-Ahead-Preise DE-LU von **Bundesnetzagentur | SMARD.de** (Filter 4169, Stundenwerte, Lizenz CC BY 4.0), 2022 bis 10/2026. Jede Wochendatei liegt unverändert in `data/raw/` mit Prüfsumme im Manifest; ein zweiter Lauf lädt nichts neu (offline reproduzierbar).

In [ ]:
manifest = lies("manifest.csv", ROOT / "data" / "raw")
preise = lies("preise_clean.csv", pfad(p, "processed"))
print(f"{len(manifest)} Wochendateien, {preise.shape[0]} Stunden, Spalten: {', '.join(preise.columns)}")

### 1b · Zeitzone, Zeitumstellung, Lücken, auffällige Preise
Schlüssel ist die **UTC-Stunde**: In UTC gibt es keine doppelte oder fehlende Stunde. Ortszeit ist nur Zusatz. Die Nächte der Zeitumstellung haben deshalb 7 bzw. 9 echte Stunden und werden genau so gerechnet. Auffällige Preise (modifizierter z-Wert > 3,5 je Jahr × Monat × Stunde) werden nur markiert, nie gelöscht.

In [ ]:
zu = lies("zeitumstellung.csv", R2)
kq = pd.DataFrame({"Kennzahl": ["Stunden", "Lücken", "imputiert", "auffällig markiert", "negative Preise", "Zeitumstellungstage"],
                   "Wert": [len(preise), int(preise['preis_eur_mwh'].isna().sum()), int(preise['imputed'].sum()),
                            f"{int(preise['flag_outlier'].sum())} ({preise['flag_outlier'].mean():.1%})",
                            f"{int((preise['preis_eur_mwh'] < 0).sum())} ({(preise['preis_eur_mwh'] < 0).mean():.1%})", len(zu)]})
kq

In [ ]:
text(f"**Interpretation:** Die Daten sind lückenlos ({len(preise)} Stunden), nichts musste ergänzt werden. "
     f"{preise['flag_outlier'].mean():.1%} der Stunden sind auffällig (z. B. Krise 2022, negative Preise bei viel Solar); sie bleiben drin, "
     "weil sie echte Marktpreise sind. Ob das Ergebnis an ihnen hängt, prüft die Sensitivität (Abschnitt 8, Variante V2).")

## 2 · Beschreibung der Preise: Wo liegt das Preistal der Nacht?

In [ ]:
jahr = lies("reinigungsfenster_jahr.csv", R2)
jahr[["jahr", "fenster", "vollstaendig", "mittel_eur_mwh", "median_eur_mwh", "eur_pro_jahr"]]

In [ ]:
zeige("g1_preistal_nacht.png")

In [ ]:
zeige("g8_heatmap_stunde_monat.png")

In [ ]:
zeige("g15_scatter_22_gegen_3.png")

In [ ]:
m = lies("blockkosten_nacht.csv"); m["jahr"] = pd.to_datetime(m["nacht"]).dt.year
t = m[m["jahr"] >= 2025].groupby("start_lokal")["preis_eur_mwh"].mean()
oracle_start = lies("backtest_naechte.csv").query("strategie == 'C' and periode in ['test_1_stunde','test_2_viertel']")["start_lokal"].value_counts(normalize=True)
text(f"**Interpretation:** In allen Zeiträumen ist der Block ab 22 Uhr der teuerste und der Block ab 3 Uhr (fast) der billigste. "
     f"Im Test 2025–2026 kostet 22–24 Uhr im Mittel {de(t[22], 1)} €/MWh, 3–5 Uhr {de(t[3], 1)} €/MWh: "
     f"{de(t[22] - t[3], 1)} €/MWh Unterschied, bei 8 MWh rund {de(8 * (t[22] - t[3]))} € pro Nacht. "
     f"Mit perfekter Voraussicht liegt der billigste Block in {oracle_start.get(3, 0):.0%} der Testnächte bei 3 Uhr, "
     f"in {oracle_start.get(2, 0):.0%} bei 2 Uhr: Das Tal ist stabil, wandert aber von Nacht zu Nacht etwas. "
     f"Die Heatmap zeigt den Rahmen: Im Winter ist die Nacht das Preistal des Tages, im Sommer liegt das Tal mittags (Solar); "
     f"3 Uhr ist aber in jedem Monat billiger als 22 Uhr. Im Streudiagramm liegt fast jede Nacht unter der Diagonalen. "
     f"Rückblickend wären mit perfekter Voraussicht je Jahr {de(jahr.loc[jahr['jahr'] == 2025, 'eur_pro_jahr'].iloc[0])} € (2025) "
     f"bis {de(jahr.loc[jahr['jahr'] == 2022, 'eur_pro_jahr'].iloc[0])} € (Krise 2022) drin gewesen (Obergrenze, nur Strom).")

## 3 · Backtest (R3 Susan)

Jeden **Freitag 08:00** plant jede Strategie die 7 Nächte Mo–So der Folgewoche, nur mit den Preisen, die dann bekannt sind (bis Freitag 24 Uhr; `src/datenzugriff.py` bricht bei Zukunftsdaten ab). Danach wird mit den echten Preisen nachgerechnet (rollierender Ursprung). Die Sonntagsnacht reicht bis Montag 06:00 und zählt mit.

| | Strategie | Regel |
| --- | --- | --- |
| 0 | **A** Status quo | immer 22–24 Uhr |
| 1 | **D** feste Regel (FS1) | bester fester Start aus dem Training 2023–2024 |
| 2 | **B_16** Wochenplanung (FS2) | je Wochentag Mittel der letzten 16 gleichen Nächte, billigster Start |
| 2′ | **B** Wochenplanung kurz | je Wochentag Median der letzten 4 gleichen Nächte |
| 3 | **C** Oracle = Tagesplanung | billigster Block mit den echten Preisen (Obergrenze) |

Die Regeln wurden nur auf dem Training entwickelt und in Commit `1d59427` eingefroren, bevor ein Testzeitraum gerechnet wurde (Hypothesen vorab in `docs/hypothesen.md`). B_16 kam nach dem Einfrieren aus dem Teamdokument dazu und ist deshalb nicht konfirmatorisch (E-049).

In [ ]:
lies("saisonaler_block.csv")[["saison", "naechte", "start_lokal", "mittel_eur_mwh", "mittel_start_22", "mittel_start_02", "mittel_start_03", "mittel_start_04"]]

In [ ]:
n = lies("backtest_naechte.csv").query("periode in ['test_1_stunde','test_2_viertel']")
wahl = pd.crosstab(n["strategie"], n["start_lokal"], normalize="index").mul(100).reindex(columns=[22, 23, 0, 1, 2, 3, 4], fill_value=0)
wahl.loc[["A", "D", "B", "B_16", "C"]].round(0)

In [ ]:
zeige("g9_heatmap_startwahl.png")

In [ ]:
text(f"**Interpretation:** Im Training ist 3 Uhr in jeder Saison der beste feste Start; eine Regel je Saison bringt deshalb nichts (D_saison = D). "
     f"Die Tabelle zeigt, wo die Strategien im Test starten (in % der Nächte): D immer 3 Uhr, B_16 meist 2 oder 3 Uhr, "
     f"das Oracle verteilt sich über 1–4 Uhr. Eine Wochenplanung weicht also oft von 3 Uhr ab, ob das etwas bringt, zeigt Abschnitt 4.")

## 4 · Ergebnisse im Test 2025–2026 (FS1, FS2, FS3)

Alles hier ist out-of-sample. Ersparnis gegenüber A in € pro Woche, **nur Strom**, alle verschobenen MWh am Spot (Faktor 1,0); Konfidenzintervall 95 % (t) und als Grenze das Block-Bootstrap-Intervall (Wochen hängen über die Saison zusammen).

In [ ]:
ki = lies("ki_strategien.csv")
ki.query("vertrag == 'band' and periode in ['test_1_stunde','test_2_viertel','test_gesamt']")[
    ["periode", "strategie", "n_wochen", "mittel_eur_woche", "ki_unten", "ki_oben", "boot_unten", "boot_oben", "eur_jahr", "anteil_oracle"]
].assign(anteil_oracle=lambda d: (100 * d["anteil_oracle"]).round(1))

In [ ]:
zeige("g7_kumuliert_test.png")

In [ ]:
g = ki.query("vertrag == 'band' and periode == 'test_gesamt'").set_index("strategie")
f3 = ki.query("vertrag == 'band' and periode == 'test_2_viertel'").set_index("strategie")
text(f"**Interpretation:** **FS1:** Fest 3 Uhr spart {de(g.loc['D','mittel_eur_woche'])} € pro Woche "
     f"(KI {de(g.loc['D','ki_unten'])}–{de(g.loc['D','ki_oben'])}), hochgerechnet {de(g.loc['D','eur_jahr'])} € pro Jahr, "
     f"das sind {g.loc['D','anteil_oracle']:.0%} des Möglichen. **FS2:** Jede Woche neu aus 16 Wochen spart {de(g.loc['B_16','mittel_eur_woche'])} € "
     f"({g.loc['B_16','anteil_oracle']:.0%}), also praktisch gleich viel; die kurze Variante B {de(g.loc['B','mittel_eur_woche'])} €. "
     f"**FS3:** Im letzten Jahr (10/2025–09/2026) hätte die Wochenplanung {de(f3.loc['B_16','mittel_eur_woche'])} € pro Woche gespart, "
     f"rund {de(f3.loc['B_16','eur_jahr'])} € im Jahr. Die Linien von D und B_16 laufen fast deckungsgleich; "
     f"der Abstand zur Tagesplanung entsteht in Nächten, deren Preistal eine Woche vorher nicht absehbar ist.")

### Tests (vorab festgelegt, einseitig, α = 0,05/3 ≈ 0,0167)

Haupttest gepaarter t-Test auf den Wochendifferenzen; Shapiro-Wilk vorher, Wilcoxon als Gegenprobe. Testentscheidung nur in der gemeinsamen Teststichprobe (89 Wochen). H0a (nach Personalkosten) folgt in Abschnitt 7.

In [ ]:
ht = lies("hypothesentests.csv").query("periode == 'test_gesamt' or hypothese == 'FS3'")
def kurs(r):
    if pd.isna(r["t"]):
        return "keine Differenz (identische Regeln)"
    pt = "< 0.001" if r["p_t"] < 0.001 else f"= {r['p_t']:.3f}"
    pw = "< 0.001" if r["p_wilcoxon"] < 0.001 else f"= {r['p_wilcoxon']:.3f}"
    return f"t({int(r['n_wochen']) - 1}) = {r['t']:.2f}; p {pt}; Wilcoxon p {pw}"
ht.assign(ergebnis=ht.apply(kurs, axis=1))[["hypothese", "vergleich", "periode", "mittel_eur_woche", "ergebnis", "h0_verworfen", "untere_schranke"]]

In [ ]:
zeige("g10_verteilung_wochen.png")

In [ ]:
zeige("g11_differenz_b_minus_d.png")

In [ ]:
b = ht.query("hypothese == 'H0b' and periode == 'test_gesamt'").iloc[0]
c = ht.query("hypothese == 'H0c' and periode == 'test_gesamt'").iloc[0]
stunden = 7 * p["kosten"]["mitarbeiter"] * p["last"]["blocklaenge_h"]
text(f"**Interpretation:** **H0b wird nicht verworfen:** Die wöchentliche Prognose B spart im Mittel {de(-b['mittel_eur_woche'])} € pro Woche "
     f"*weniger* als fest 3 Uhr (p = {b['p_t']:.2f}); Wilcoxon kommt zum selben Ergebnis. Eine Prognose braucht es nicht. "
     f"**H0c (Kipppunkt):** Tagesplanung holt {de(c['mittel_eur_woche'])} € pro Woche mehr als B; untere Schranke {de(c['untere_schranke'])} € pro Woche. "
     f"Sie lohnt sich nur, wenn kurzfristig gebuchtes Personal weniger als {de(c['untere_schranke'] / stunden, 2)} € pro Personalstunde Aufpreis kostet. "
     f"FS1–FS3 (jede Strategie gegen 22–24 Uhr) sind klar signifikant. Shapiro-Wilk lehnt die Normalverteilung der Wochendifferenzen ab; "
     f"bei 89 Wochen trägt der t-Test trotzdem, und der Wilcoxon bestätigt jede Entscheidung. "
     "Die Verteilungen zeigen, warum: Die Wochen streuen rechtsschief (einzelne sehr teure Wochen), "
     "und die Differenz B − D liegt eng um null mit breiten Rändern, wie der QQ-Plot zeigt.")

## 6 · Jahreszeiten und Fehleranalyse: Wann liegt die Wochenplanung daneben? (R3)

Bedauern (Regret) = Kosten B − Kosten C je Nacht; B − D = was die kurze Prognose gegenüber fest 3 Uhr gewinnt (< 0) oder verliert (> 0).

In [ ]:
fa = lies("fehleranalyse_b.csv")
fa.query("periode in ['test_1_stunde','test_2_viertel'] and gruppe == 'saison'").pivot_table(
    index="wert", columns="periode", values=["regret_b_eur_nacht", "b_minus_d_eur_nacht", "anteil_b_trifft_oracle_pct"]).round(1)

In [ ]:
s = fa.query("periode in ['test_1_stunde','test_2_viertel'] and gruppe == 'saison'")
text(f"**Interpretation:** B trifft den billigsten Block nur in {s['anteil_b_trifft_oracle_pct'].min():.0f}–{s['anteil_b_trifft_oracle_pct'].max():.0f} % der Nächte. "
     "In den meisten Saisons verliert B gegenüber fest 3 Uhr ein paar Euro pro Nacht, gewinnt aber nirgends deutlich: "
     "Die Abweichungen von 3 Uhr folgen Zufallsschwankungen der letzten Wochen, nicht einem Muster, das sich eine Woche vorher erkennen lässt.")

## 7 · Personalkosten und Wirtschaftlichkeit (R4 Caro)

Personal je Reinigung: 3 Personen × 35 €/h × 2 h × (1 + Zuschlag). Zuschlag je Stunde nach dem **Kalendertag der Stunde** (Sa 22 Uhr = Samstag, So 2 Uhr = Sonntag): Nacht 25 % (22–06 Uhr), Wochenende 50 %, Feiertag 100 %, nur der höchste. Netto = Stromersparnis × Vertragsfaktor − Personal-Mehrkosten gegenüber A. **H0a:** Die Netto-Ersparnis von D ist ≤ 0.

In [ ]:
nt = lies("netto_tests.csv")
nt.query("periode in ['test_gesamt','stresstest']")[["strategie", "periode", "vertrag", "strom_ersparnis_eur_woche", "personal_mehr_eur_woche",
                                                      "netto_eur_woche", "ki_unten", "ki_oben", "netto_eur_jahr", "p_t", "h0_verworfen"]]

In [ ]:
zeige("g2_ersparnis_strategien.png")

In [ ]:
zeige("g12_wasserfall.png")

In [ ]:
lies("kipppunkte.csv")[["vertrag", "annahme", "basis", "kipppunkt", "einheit"]]

In [ ]:
zeige("g5_break_even.png")

In [ ]:
zeige("g3_kennzahl_zeitverlauf.png")

In [ ]:
zeige("g13_kalender_ersparnis.png")

In [ ]:
d = nt.query("strategie == 'D' and periode == 'test_gesamt'").set_index("vertrag")
kp = lies("kipppunkte.csv").query("einheit == 'Stunden je Person und Nacht'").set_index("vertrag")["kipppunkt"]   # paid waiting hours
be = lies("wirtschaftlichkeit.csv").query("strategie == 'D' and periode == 'test_gesamt' and umstellungskosten_eur == 25000").set_index("vertrag")
kpi = lies("kpi_rollierend.csv")
text(f"**Interpretation:** **H0a wird verworfen:** Fest 3 Uhr spart auch nach Personalkosten {de(d.loc['band','netto_eur_woche'])} € pro Woche "
     f"({de(d.loc['band','netto_eur_jahr'])} € pro Jahr; wenn nur 30 % der MWh am Spot zählen {de(d.loc['proportional','netto_eur_jahr'])} €), p < 0,001. "
     f"Die Zuschläge spielen kaum eine Rolle ({de(d.loc['band','personal_mehr_eur_woche'])} € pro Woche), weil das ganze Fenster ohnehin Nacht ist "
     "und sich Wochenendstunden über die Woche ausgleichen. In keiner Woche 2022–2026 war 3 Uhr teurer als 22 Uhr "
     f"(kleinstes 13-Wochen-Mittel {de(kpi['netto_rollierend_13w'].min())} €). "
     f"**Der Kipppunkt ist die Organisation des Personals:** Muss jede Person je Nacht mehr als {de(kp['band'], 1)} h "
     f"(30 %-Fall {de(kp['proportional'], 1)} h) zusätzlich bezahlt werden, etwa weil sie von 22 bis 3 Uhr wartet, ist die Ersparnis weg. "
     f"25 000 € Umstellungskosten wären nach {de(be.loc['band','break_even'], 1)} Monaten (30 %-Fall {de(be.loc['proportional','break_even'], 1)}) wieder drin.")

### Warum nur die Reinigung? C&E-Matrix (Skala 0/1/3/9, Gewichte Teamannahme)

In [ ]:
lies("ce_matrix.csv").drop(columns=["begruendung"])

In [ ]:
zeige("g14_pareto_ce.png")

In [ ]:
ce = lies("ce_matrix.csv").iloc[1:]
text(f"**Interpretation:** Die Reinigung erreicht {ce.iloc[0]['anteil_pct']:.0f} % der gewichteten Punkte: Sie ist im Wochenplan steuerbar, "
     "trägt kein Produktrisiko und hängt am Schichtplan. Die Produktion hat zwar die meiste Leistung, ist aber wegen der Rohmilch (36–48 h) "
     "und der festen Betriebszeit nicht verschiebbar; die Kälte läuft automatisch und braucht keinen Schichtplan. "
     "Ursachen der Lücke zum Oracle als Ishikawa: `docs/ishikawa.md`.")

## 8 · Sensitivität: Hängt das Ergebnis an Daten und Annahmen? (R2 Max)

Zwölf Varianten, Schwelle 5 % vorab festgelegt, alle berichtet. Kennzahl: Netto-Ersparnis von D im Test.

In [ ]:
sens = lies("sensitivitaet.csv", R2).query("strategie == 'D' and periode == 'test_gesamt'")
sens[["variante", "schalter", "wert", "mittel_eur_woche", "delta_pct", "anteil_oracle", "h0_entscheidung_gleich", "relevant"]].round(3)

In [ ]:
zeige("g4_sensitivitaet.png")

In [ ]:
v2 = sens.set_index("variante").loc["V2"]
text(f"**Interpretation:** Keine Variante kehrt eine Testentscheidung um, der Anteil am Oracle bleibt bei "
     f"{sens['anteil_oracle'].min():.0%}–{sens['anteil_oracle'].max():.0%}. Die Euro-Beträge hängen dagegen stark an Menge, Vertrag, "
     "Produktionstagen und Dauer der Reinigung und gehören deshalb als Bandbreite in den Bericht. "
     f"Ohne die Nächte mit auffälligen Preisen sinkt die Ersparnis um {de(-v2['delta_pct'])} %; die Aussage hängt also nicht an Extremstunden.")

## 9 · Spot vs. Festpreis (R1 Chiara)

Mittlerer Day-Ahead-Preis im Lastprofil des Werks (Annahme: 180 MWh/Tag, Reinigung 22–24 Uhr, Rest gleichmäßig 06–22 Uhr) gegen den Festpreis von 85 €/MWh.

In [ ]:
sf = lies("spot_festpreis.csv")
sf

In [ ]:
zeige("g6_spot_festpreis.png")

In [ ]:
text(f"**Interpretation:** Außer 2024 ({de(sf.loc[sf['jahr'] == 2024, 'spot_profil_eur_mwh'].iloc[0], 1)} €/MWh) lag die Börse im Lastprofil über 85 €/MWh; "
     f"2026 bis Oktober bei {de(sf.loc[sf['jahr'] == 2026, 'spot_profil_eur_mwh'].iloc[0], 1)} €/MWh. Der Festpreis schützt vor Ausreißern wie 2022. "
     "Die Verschiebung der Reinigung wirkt aber nur auf den Spotanteil; mit mehr Spot wächst ihr Wert (bis Faktor 1,0), mit mehr Festpreis schrumpft er (0,3). "
     "Der Vergleich ist vereinfacht: Ein Festpreis gilt jeweils für ein Lieferjahr und wird im Januar davor eingekauft, 85 €/MWh ist der heutige Wert.")

## 10 · Bericht

### Fazit: Empfehlung an die Werksleitung (Entwurf R1 mit R4)

In [ ]:
text(f"1. **Reinigung fest um 3 Uhr statt 22 Uhr einplanen** (Strategie D). Spart netto rund {de(d.loc['band','netto_eur_jahr'])} € pro Jahr "
     f"(nur 30 % am Spot: {de(d.loc['proportional','netto_eur_jahr'])} €), {g.loc['D','anteil_oracle']:.0%} des Möglichen.\n"
     "2. **Keine Preisprognose, keine wöchentliche Neuberechnung:** bringt nicht messbar mehr (H0b, FS2).\n"
     f"3. **Bedingung:** eigene Reinigungsschicht ohne bezahlte Wartezeit. Ab {de(kp['band'], 1)} h bezahlter Wartezeit je Person und Nacht kippt es.\n"
     "4. **Wer entscheidet:** Werksleitung (Schichtplan) mit dem zentralen Einkauf (Vertrag).\n"
     "5. **Kennzahl:** Netto-Ersparnis gegenüber 22–24 Uhr je Woche, rollierend über 13 Wochen (Stromrechnung minus Personal-Mehrkosten laut Abrechnung).\n"
     "6. **Nachsteuern:** einmal im Jahr die beste feste Uhrzeit mit den Preisen des Vorjahres neu bestimmen.\n"
     "7. **Rücknahme:** fällt die Kennzahl unter null, zurück auf 22 Uhr und Ursache klären.")

### Annahmen und offene Punkte

- **Annahmen:** vollständig in `docs/annahmen.md` (Partner, Schätzung, Team, Daten). Kritisch für die Euro-Beträge: Vertragsfaktor (1,0 oder 0,3) und Menge (8 MWh); für die Entscheidung: bezahlte Wartezeit des Personals.
- **Nichtziele:** Kälte, Produktion, Viertelstunden, Intraday, Netzentgelte (`docs/annahmen.md`).
- **Grenzen:** Wochen hängen zusammen (Bootstrap-Intervalle rund 50 % breiter als t); B_16 und FS1–FS3 nach dem Einfrieren ergänzt (E-049); Kalendermerkmale verbessern die Regel nicht (K3 negativ, E-048); D im Stresstest 2022 mit einer Regel aus 2023–2024.
- **Offen:** Partner bestätigt Zuschlagsregeln und ob die Reinigungsmannschaft warten müsste (A-017, M-004); Bestätigung der Owner im Tagebuch (E-045).
- **Belege:** `docs/projekttagebuch.md`, `docs/hypothesen.md`, `docs/erklaerzettel.md`, `ergebnisse/daten_fuer_grafiken.md`.